# 02 - Data Analysis Report (all 14 B tasks)
Every task needs a number/table/chart **and an interpretation**. The complete written report is `reports/B_analysis_report.md`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
SEED = 42
df=pd.read_csv(ROOT/'data/processed/master_train.csv')
df['revenue_per_ha']=df.yield_tons_per_ha*10*df.price_birr_per_quintal

## B1.1 Revenue per hectare by crop

In [ ]:
df.groupby('crop_type').agg(mean_yield=('yield_tons_per_ha','mean'),mean_revenue=('revenue_per_ha','mean')).round(2).sort_values('mean_revenue',ascending=False)

## B1.2 Price trend

In [ ]:
p=pd.read_csv(ROOT/'data/processed/clean_prices.csv'); q=p.groupby(['crop_type','year']).price_birr_per_quintal.mean().unstack(); out=pd.DataFrame({'2021':q[2021],'2024':q[2024]}); out['change']=out['2024']-out['2021']; out['pct_change']=100*out.change/out['2021']; out.round(2)

## B2.1-B2.3 Region and crop

In [ ]:
display(df.groupby('region').yield_tons_per_ha.agg(['mean','median','std']).round(3))
display(df.groupby('crop_type').yield_tons_per_ha.agg(['mean','median','std']).round(3))
display(df.pivot_table(index='region',columns='crop_type',values='yield_tons_per_ha').round(3))

## B2.4-B2.5 Improved seed and pest/disease

In [ ]:
for col in ['improved_seed_used','pest_disease_flag']:
    t=df.dropna(subset=[col]).groupby(['crop_type',col]).yield_tons_per_ha.mean().unstack(); t['gap']=t.get(1)-t.get(0); t['pct_gap']=100*t.gap/t.get(0); display(t.round(3))

## B3.1-B3.4 Agronomic drivers

In [ ]:
d=df.copy(); d['fert_band']=pd.qcut(d.fertilizer_kg_per_ha,4,duplicates='drop'); d['alt_band']=pd.cut(d.altitude_m,4); d['dist_band']=pd.qcut(d.distance_to_market_km,4,duplicates='drop')
display(d.groupby('fert_band',observed=True).yield_tons_per_ha.mean().round(3))
display(d.groupby(['crop_type','alt_band'],observed=True).yield_tons_per_ha.agg(['mean','count']).round(3))
display(d.pivot_table(index='planting_month',columns='crop_type',values='yield_tons_per_ha').round(3))
print('Distance correlation:',d[['distance_to_market_km','yield_tons_per_ha']].corr().iloc[0,1].round(3)); display(d.groupby('dist_band',observed=True).yield_tons_per_ha.mean().round(3))

## B4.1-B4.3 Time and weather

In [ ]:
display(df.pivot_table(index='survey_year',columns='region',values='yield_tons_per_ha').round(3))
ry=df.groupby(['region','survey_year']).agg(temp=('season_avg_temp_c','mean'),yield_mean=('yield_tons_per_ha','mean')).reset_index(); ry['region_4yr_temp']=ry.groupby('region').temp.transform('mean'); ry['temp_anomaly']=ry.temp-ry.region_4yr_temp; display(ry.round(3))
r=df[['rainfall_mm_season','season_rainfall_mm']].dropna(); print('Rainfall correlation:',r.corr().iloc[0,1].round(3)); print('Mean absolute difference:',(r.iloc[:,0]-r.iloc[:,1]).abs().mean().round(2),'mm')

## Read the final interpretations
Open `reports/B_analysis_report.md`. It contains all B1.1-B4.3 headings and the required one/two-sentence interpretations.